# Comparador de modelos predictivos: Random Forest y XGBoost
### Aplicación de machine learning para predecir la flotación de relaves de lixiviación, Lima 2026
**Barboza Duran, Edwin Miguel · Ruiz Torres, Alfonso** — Maestría en Inteligencia Artificial, Universidad César Vallejo

Versión en Python (Google Colab) de la aplicación desarrollada en R + Shiny (`S3_TESIS_MIA_BarbozaEdwin-RuizAlfonso_Codigo_RStudio_v7.R`). Mantiene el mismo protocolo, las mismas hipótesis y las mismas pestañas (Portada, Datos, Resultados, Discusión y Conclusiones):

* Partición temporal 70 / 15 / 15 (entrenamiento / validación / prueba).
* Modelos ajustados con entrenamiento y evaluados en validación; reentrenados con entrenamiento + validación y evaluados una sola vez en prueba.
* Modelo base: media histórica del conjunto de ajuste.
* Las dosificaciones de los cinco reactivos son predictores de las tres variables; el consumo específico total es predictor de la recuperación y de la ley.
* Hipótesis Ho-Hi y Ho1-Hi1 a Ho3-Hi3: se acepta Hi si R² ≥ 0,75, MAPE ≤ 10 % y MAE y RMSE menores que los del modelo base.
* Hipótesis Ho4-Hi4: se acepta Hi4 si un modelo tiene mayor R² y menor MAPE, MAE y RMSE que el otro, y la prueba Diebold-Mariano (HLN) da p < 0,05 con errores al cuadrado y con errores absolutos.

**Cómo usarlo**

1. Menú **Entorno de ejecución → Ejecutar todo** (o ejecuta las celdas en orden).
2. En el **Paso 1** elige si el Excel se sube desde tu computadora o se lee desde Google Drive.
3. En el **Paso 2** revisa las opciones y ejecuta el protocolo.
4. El **Paso 3** muestra las pestañas *Portada*, *Datos*, *Resultados*, *Discusión* y *Conclusiones*; el **Paso 4** genera y descarga el informe PDF.

In [ ]:
#@title 0 · Instalación e importación de librerías
import importlib, importlib.util, subprocess, sys
for paquete, modulo in [("xgboost", "xgboost"), ("openpyxl", "openpyxl")]:
    if importlib.util.find_spec(modulo) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paquete], check=True)

import io, os, re, glob, textwrap, unicodedata, warnings
from datetime import datetime
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.patches import Rectangle
from IPython.display import display, HTML
warnings.filterwarnings("ignore")

try:
    import google.colab  # noqa: F401
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

print(f"Entorno: {'Google Colab' if EN_COLAB else 'Jupyter local'} · pandas {pd.__version__} · "
      f"scikit-learn {importlib.import_module('sklearn').__version__} · xgboost {xgb.__version__}")

In [ ]:
#@title 1 · Configuración del protocolo (diccionario, umbrales e hiperparámetros de la tesis)
ETIQUETAS_TESIS = {
    "masa_alimentacion": "Masa de alimentación (t)", "ley_alimentacion_au": "Ley de Au de cabeza (g/t)",
    "densidad_pulpa": "Densidad de pulpa (g/L)", "pct_solidos": "Porcentaje de sólidos (%)",
    "granulometria": "Granulometría -200 mallas (%)", "ph_inicial": "pH inicial", "ph_final": "pH final",
    "ph": "pH", "potencial_redox": "Potencial redox (mV)", "caudal_alimentacion": "Caudal de pulpa (m³/h)",
    "tiempo_residencia": "Tiempo de flotación (min)", "sulfato_cobre": "Sulfato de cobre (kg/t)",
    "metabisulfito_sodio": "Metabisulfito de sodio (kg/t)", "pax_sipx": "PAX + SIPX (Z-6 + Z-11) (kg/t)",
    "mibc": "MIBC (kg/t)", "antiespumante": "Antiespumante (kg/t)",
    "recuperacion_au": "Recuperación de Au (%)", "ley_concentrado_au": "Ley de Au en el concentrado (g/t)",
    "consumo_reactivos": "Consumo específico de reactivos (kg/t)",
}
OBJETIVOS_TESIS = ["recuperacion_au", "ley_concentrado_au", "consumo_reactivos"]
NUMERACION = {"recuperacion_au": dict(tabla=5, pred=2, imp=3),
              "ley_concentrado_au": dict(tabla=6, pred=4, imp=5),
              "consumo_reactivos": dict(tabla=7, pred=6, imp=7)}
UMBRAL_R2, UMBRAL_MAPE, UMBRAL_P = 0.75, 10.0, 0.05
REACTIVOS_TESIS = ["sulfato_cobre", "metabisulfito_sodio", "pax_sipx", "mibc", "antiespumante"]
PATRON_META = r"^(a.o|anio|year|mes|month|semana|week|dia|day|id|codigo|indice|index|orden|fila)$"

# Hiperparámetros por objetivo (max_depth None = sin límite)
HP_TESIS = {
    "recuperacion_au": {
        "rf":  dict(ntree=300, max_depth=8, nodesize=5, max_features=0.6),
        "xgb": dict(nrounds=500, eta=0.01, max_depth=3, subsample=0.9, colsample=0.9)},
    "ley_concentrado_au": {
        "rf":  dict(ntree=300, max_depth=None, nodesize=1, max_features=1.0),
        "xgb": dict(nrounds=200, eta=0.10, max_depth=5, subsample=0.7, colsample=0.9)},
    "consumo_reactivos": {
        "rf":  dict(ntree=300, max_depth=8, nodesize=10, max_features=0.6),
        "xgb": dict(nrounds=200, eta=0.03, max_depth=3, subsample=0.7, colsample=0.6)},
}
HP_GENERICOS = {"rf":  dict(ntree=300, nodesize=5, max_features=0.6, max_depth=None),
                "xgb": dict(nrounds=200, eta=0.05, max_depth=3, subsample=0.8, colsample=0.8)}

VERDE = "#0F6E56"
COLORES_MODELO = {"Random Forest": "#2A78D6", "XGBoost": "#EB6834", "Modelo base": "#D1D5DB"}
COLORES_SUBCONJUNTO = {"Entrenamiento": "#2A78D6", "Validación": "#EB6834", "Prueba": "#1BAF7A"}
SUBCONJUNTOS = ["Entrenamiento", "Validación", "Prueba"]

def rango_r2(x):
    if not np.isfinite(x): return "—"
    return "Sustancial" if x >= 0.75 else "Moderado" if x >= 0.5 else "Débil" if x >= 0.25 else "Insuficiente"
def rango_mape(x):
    if not np.isfinite(x): return "—"
    return "Excelente" if x < 10 else "Bueno" if x < 20 else "Aceptable" if x < 50 else "Inexacto"
def rango_rrmse(x):
    if not np.isfinite(x): return "—"
    return "Excelente" if x < 10 else "Bueno" if x < 20 else "Regular" if x < 30 else "Deficiente"
print("Configuración cargada.")

In [ ]:
#@title 2 · Funciones de lectura y limpieza de datos
ALIAS_COLUMNAS = {
    "fecha": "Fecha", "mes": "Mes", "anio": "anio", "ano": "anio", "guardia": "Guardia",
    "lauccpygt": "ley_concentrado_au", "na2s2o5kgt": "metabisulfito_sodio",
    "masadesolidostms": "masa_alimentacion", "laucbzgtt": "ley_alimentacion_au",
    "laucbzgt": "ley_alimentacion_au", "densidaddepulpagl": "densidad_pulpa",
    "solidos": "pct_solidos", "granulometria200mesh": "granulometria",
    "phinic": "ph_inicial", "phfin": "ph_final", "ph": "ph", "orpmv": "potencial_redox",
    "qpm3h": "caudal_alimentacion", "qp": "caudal_alimentacion",
    "tiempoflotmin": "tiempo_residencia", "cuso45h2okgt": "sulfato_cobre",
    "nas2o5kgt": "metabisulfito_sodio", "z6z11kgt": "pax_sipx", "mibckgt": "mibc",
    "antiespumantekgt": "antiespumante", "rauccpy": "recuperacion_au",
    "lauccpy": "ley_concentrado_au", "consumoespecificototalkgt": "consumo_reactivos",
    "particionmodelado": "Particion", "particion": "Particion",
}

def limpiar_texto(x):
    """Quita saltos de línea, tabulaciones y espacios dobles."""
    return re.sub(r" {2,}", " ", re.sub(r"[\r\n\t]+", " ", str(x))).strip()

def normalizar_clave(x):
    """Minúsculas, sin tildes ni símbolos: 'L.Au ccPy (g/t)' -> 'lauccpygt'."""
    s = unicodedata.normalize("NFKD", limpiar_texto(x).lower()).encode("ascii", "ignore").decode()
    return re.sub(r"[^a-z0-9]+", "", s)

def _nombres_unicos(nombres):
    vistos, salida = {}, []
    for n in nombres:
        if n in vistos:
            vistos[n] += 1; salida.append(f"{n}.{vistos[n]}")
        else:
            vistos[n] = 0; salida.append(n)
    return salida

def normalizar_columnas(df):
    """Traduce los encabezados de planta a los nombres internos del modelo."""
    df = df.copy()
    df.columns = _nombres_unicos([ALIAS_COLUMNAS.get(normalizar_clave(c), limpiar_texto(c)) for c in df.columns])
    return df

def leer_excel_robusto(origen):
    df = pd.read_excel(origen, sheet_name=0)
    if df.empty: raise ValueError("La primera hoja del Excel está vacía.")
    vacias = [i for i, c in enumerate(df.columns)
              if pd.isna(c) or str(c).startswith("Unnamed:") or not str(c).strip()]
    if vacias and vacias[0] > 0:          # corta columnas basura a la derecha del rango útil
        df = df.iloc[:, :vacias[0]]
    return normalizar_columnas(df)

def leer_csv_robusto(origen):
    crudo = origen.read() if hasattr(origen, "read") else open(origen, "rb").read()
    try:
        texto = crudo.decode("utf-8-sig")
    except UnicodeDecodeError:
        texto = crudo.decode("latin-1")
    primera = texto.splitlines()[0] if texto else ""
    if primera.count(";") > primera.count(","):
        df = pd.read_csv(io.StringIO(texto), sep=";", decimal=",")
    else:
        df = pd.read_csv(io.StringIO(texto))
    return normalizar_columnas(df)

def leer_archivo_robusto(origen, nombre):
    ext = os.path.splitext(nombre)[1].lower()
    return leer_excel_robusto(origen) if ext in (".xlsx", ".xls", ".xlsm") else leer_csv_robusto(origen)

def etiqueta(col):
    return ETIQUETAS_TESIS.get(col, col)

def columnas_numericas(df):
    """Columnas numéricas candidatas (excluye año, mes, índices, etc.)."""
    return [c for c in df.columns
            if pd.api.types.is_numeric_dtype(df[c]) and not pd.api.types.is_bool_dtype(df[c])
            and not re.match(PATRON_META, str(c).lower())]

def normalizar_particion(serie):
    s = serie.astype(str).map(lambda t: unicodedata.normalize("NFKD", t.strip().lower())
                              .encode("ascii", "ignore").decode())
    return np.select([s.str.match(r"^(entren|train|trai)"), s.str.match(r"^(valid|vali)"),
                      s.str.match(r"^(prueba|test)")],
                     ["Entrenamiento", "Validación", "Prueba"], default=None)

def resumen_carga(df, nombre):
    print(f"Archivo cargado: {nombre} · {len(df)} registros × {df.shape[1]} columnas.")
    if "Particion" in df.columns:
        print("Columna Particion:", pd.Series(normalizar_particion(df["Particion"])).value_counts().to_dict())
    cols = columnas_numericas(df)
    faltantes = (df[cols].isna().mean() * 100).round(1)
    tabla = pd.DataFrame({"Columna": cols, "Etiqueta": [etiqueta(c) for c in cols],
                          "% faltantes": faltantes.values,
                          "Rol": ["Objetivo" if c in OBJETIVOS_TESIS else "Predictor" for c in cols]})
    print(f"Variables numéricas detectadas: {len(cols)} "
          f"({sum(c in OBJETIVOS_TESIS for c in cols)} objetivos de la tesis).")
    return tabla

In [ ]:
#@title 3 · Funciones de métricas, prueba Diebold-Mariano y criterios
def calcular_metricas(real, pred):
    real = np.asarray(real, float); pred = np.asarray(pred, float)
    ok = np.isfinite(real) & np.isfinite(pred); real, pred = real[ok], pred[ok]
    if len(real) == 0:
        return dict(R2=np.nan, MAE=np.nan, RMSE=np.nan, MAPE=np.nan, RRMSE=np.nan, n=0)
    e = real - pred
    rmse, mae = float(np.sqrt(np.mean(e ** 2))), float(np.mean(np.abs(e)))
    ss_tot = np.sum((real - real.mean()) ** 2)
    r2 = float(1 - np.sum(e ** 2) / ss_tot) if ss_tot > np.finfo(float).eps else np.nan
    tol = max(1e-8, np.max(np.abs(real)) * 1e-6)
    validos = np.abs(real) > tol
    mape = float(np.mean(np.abs(e[validos] / real[validos])) * 100) if validos.any() else np.nan
    media = real.mean()
    rrmse = float(100 * rmse / abs(media)) if abs(media) > tol else np.nan
    return dict(R2=r2, MAE=mae, RMSE=rmse, MAPE=mape, RRMSE=rrmse, n=int(len(real)))

def prueba_dm(e1, e2, h=1, perdida="cuadratica"):
    """Diebold-Mariano con corrección de Harvey-Leybourne-Newbold.
    perdida = "cuadratica" (e²) o "absoluta" (|e|). DM > 0: Random Forest tiene más error."""
    e1, e2 = np.asarray(e1, float), np.asarray(e2, float)
    ok = np.isfinite(e1) & np.isfinite(e2)
    L = np.square if perdida == "cuadratica" else np.abs
    d = L(e1[ok]) - L(e2[ok]); n = len(d)
    if n < 3 or h < 1 or h >= n:
        return dict(estadistico=np.nan, p_valor=np.nan, n=n, interpretacion="Muestra insuficiente")
    var_d = np.var(d, ddof=1)
    if not np.isfinite(var_d) or var_d <= np.finfo(float).eps:
        return dict(estadistico=np.nan, p_valor=np.nan, n=n, interpretacion="Varianza nula")
    dm = d.mean() / np.sqrt(var_d / n)
    dm_hln = dm * np.sqrt((n + 1 - 2 * h + h * (h - 1) / n) / n)
    p = float(2 * stats.t.cdf(-abs(dm_hln), df=n - 1))
    return dict(estadistico=float(dm_hln), p_valor=p, n=n,
                interpretacion="Diferencia significativa" if p < 0.05 else "Sin diferencia significativa")

def evaluar_criterios(m, base):
    """Ho1-Hi1 a Ho3-Hi3: R² ≥ 0,75, MAPE ≤ 10 % y MAE y RMSE menores que los del modelo base."""
    vals = [m["R2"], m["MAPE"], m["MAE"], m["RMSE"], base["MAE"], base["RMSE"]]
    ok = (all(np.isfinite(v) for v in vals) and m["R2"] >= UMBRAL_R2 and m["MAPE"] <= UMBRAL_MAPE
          and m["MAE"] < base["MAE"] and m["RMSE"] < base["RMSE"])
    return "Sí" if ok else "No"

def descriptivos_por_subconjunto(particion, variables):
    filas = []
    for nombre, d in zip(SUBCONJUNTOS, (particion["train"], particion["valid"], particion["test"])):
        for v in variables:
            bruto = pd.to_numeric(d[v], errors="coerce").to_numpy(float)
            z = bruto[np.isfinite(bruto)]
            filas.append(dict(
                Subconjunto=nombre, Variable=v, n=len(z), n_NA=int((~np.isfinite(bruto)).sum()),
                Media=z.mean() if len(z) else np.nan, DE=z.std(ddof=1) if len(z) > 1 else np.nan,
                Min=z.min() if len(z) else np.nan, Q1=np.quantile(z, 0.25) if len(z) else np.nan,
                Mediana=np.median(z) if len(z) else np.nan, Q3=np.quantile(z, 0.75) if len(z) else np.nan,
                Max=z.max() if len(z) else np.nan,
                CV_pct=100 * z.std(ddof=1) / abs(z.mean()) if len(z) > 1 and z.mean() != 0 else np.nan))
    return pd.DataFrame(filas)

In [ ]:
#@title 4 · Núcleo del algoritmo: imputación, modelos, partición e hipótesis
def calcular_imputaciones(referencia, predictores, metodo="media"):
    valores = {}
    for v in predictores:
        z = pd.to_numeric(referencia[v], errors="coerce").to_numpy(float); z = z[np.isfinite(z)]
        valores[v] = 0.0 if (metodo == "cero" or len(z) == 0) else float(z.mean())
    return valores

def imputar(d, predictores, referencia, metodo):
    """Rellena faltantes con la media (o 0) calculada SOLO en el conjunto de referencia."""
    valores = calcular_imputaciones(referencia, predictores, metodo)
    X = pd.DataFrame(index=d.index)
    for v in predictores:
        col = pd.to_numeric(d[v], errors="coerce").astype(float)
        if col.isna().all() and d[v].notna().any():
            raise ValueError(f"La variable '{v}' no es numérica ni convertible.")
        X[v] = col.where(np.isfinite(col), np.nan).fillna(valores[v])
    return X

def ajustar_rf(X, y, hp):
    """Random Forest equivalente a randomForest de R: mtry, nodesize y maxnodes = 2^max_depth."""
    p = X.shape[1]; mtry = max(1, int(np.floor(hp["max_features"] * p)))
    md = hp.get("max_depth")
    max_hojas = None if (md is None or md <= 0) else max(2, int(2 ** min(md, 20)))
    modelo = RandomForestRegressor(n_estimators=hp["ntree"], max_features=mtry,
                                   min_samples_split=hp["nodesize"] + 1, max_leaf_nodes=max_hojas,
                                   bootstrap=True, random_state=42, n_jobs=-1)
    return modelo.fit(np.asarray(X, float), np.asarray(y, float))

def ajustar_xgb(X, y, hp):
    dtrain = xgb.DMatrix(np.asarray(X, float), label=np.asarray(y, float))
    params = dict(objective="reg:squarederror", eta=hp["eta"], max_depth=hp["max_depth"],
                  subsample=hp["subsample"], colsample_bytree=hp["colsample"], nthread=1, seed=42)
    return xgb.train(params, dtrain, num_boost_round=hp["nrounds"])

def predecir(modelo, X):
    X = np.asarray(X, float)
    return modelo.predict(xgb.DMatrix(X)) if isinstance(modelo, xgb.Booster) else modelo.predict(X)

def predictores_para(objetivo, predictores, columnas):
    """Dosificaciones siempre; consumo total para recuperación y ley; nunca el propio objetivo."""
    p = list(dict.fromkeys(list(predictores) + [c for c in REACTIVOS_TESIS if c in columnas]))
    if "consumo_reactivos" in columnas and "consumo_reactivos" not in p:
        p.append("consumo_reactivos")
    return [c for c in p if c != objetivo]

def entrenar_evaluar_target(objetivo, predictores, part, usar_hp_tesis=True, metodo_na="media"):
    ok = lambda d: d[np.isfinite(pd.to_numeric(d[objetivo], errors="coerce").astype(float))]
    train, valid, test = ok(part["train"]), ok(part["valid"]), ok(part["test"])
    if len(train) < 5: raise ValueError(f"'{objetivo}': menos de 5 registros de entrenamiento.")
    if len(test) < 2: raise ValueError(f"'{objetivo}': menos de 2 registros de prueba.")
    hp = HP_TESIS.get(objetivo) if usar_hp_tesis else None
    hp_rf, hp_xgb = (hp or HP_GENERICOS)["rf"], (hp or HP_GENERICOS)["xgb"]
    y = lambda d: pd.to_numeric(d[objetivo]).astype(float).to_numpy()

    # Fase 1: ajuste con entrenamiento, evaluación en entrenamiento y validación
    x_tr, x_va = imputar(train, predictores, train, metodo_na), imputar(valid, predictores, train, metodo_na)
    rf1, xg1 = ajustar_rf(x_tr, y(train), hp_rf), ajustar_xgb(x_tr, y(train), hp_xgb)
    filas = []
    agregar = lambda mod, sub, real, pred: filas.append(dict(Modelo=mod, Subconjunto=sub, **calcular_metricas(real, pred)))
    agregar("Random Forest", "Entrenamiento", y(train), predecir(rf1, x_tr))
    agregar("XGBoost", "Entrenamiento", y(train), predecir(xg1, x_tr))
    if len(valid):
        agregar("Random Forest", "Validación", y(valid), predecir(rf1, x_va))
        agregar("XGBoost", "Validación", y(valid), predecir(xg1, x_va))

    # Fase 2: reentreno con entrenamiento + validación, una sola predicción del conjunto de prueba
    ajuste = pd.concat([train, valid])
    x_aj, x_te = imputar(ajuste, predictores, ajuste, metodo_na), imputar(test, predictores, ajuste, metodo_na)
    y_aj, real_test = y(ajuste), y(test)
    rf2, xg2 = ajustar_rf(x_aj, y_aj, hp_rf), ajustar_xgb(x_aj, y_aj, hp_xgb)
    pred = {"Random Forest": predecir(rf2, x_te), "XGBoost": predecir(xg2, x_te)}
    agregar("Modelo base", "Prueba", real_test, np.full(len(real_test), y_aj.mean()))
    agregar("Random Forest", "Prueba", real_test, pred["Random Forest"])
    agregar("XGBoost", "Prueba", real_test, pred["XGBoost"])
    metricas_todas = pd.DataFrame(filas)

    m = metricas_todas[metricas_todas.Subconjunto == "Prueba"].drop(columns="Subconjunto").reset_index(drop=True)
    base = m[m.Modelo == "Modelo base"].iloc[0]
    m["Red_MAE_vs_base"] = 100 * (base.MAE - m.MAE) / base.MAE if base.MAE > 0 else np.nan
    m["Red_RMSE_vs_base"] = 100 * (base.RMSE - m.RMSE) / base.RMSE if base.RMSE > 0 else np.nan
    media_obs = float(real_test.mean())
    m["Cumple_criterios"] = [("—" if r.Modelo == "Modelo base" else evaluar_criterios(r, base))
                             for _, r in m.iterrows()]
    m["Rango_R2"], m["Rango_MAPE"], m["Rango_RRMSE"] = m.R2.map(rango_r2), m.MAPE.map(rango_mape), m.RRMSE.map(rango_rrmse)

    # Ranking: suma de puestos en RMSE, MAPE y R² (el menor gana)
    cand = m[m.Modelo != "Modelo base"]; n_c = len(cand)
    ranking = (cand.RMSE.rank(method="min").fillna(n_c + 1) + cand.MAPE.rank(method="min").fillna(n_c + 1)
               + (-cand.R2).rank(method="min").fillna(n_c + 1))
    m["Ranking"] = ranking.reindex(m.index)
    mejor = m.loc[ranking.idxmin(), "Modelo"]
    m["EsMejor"] = m.Modelo == mejor

    return dict(objetivo=objetivo, metricas=m, metricas_todas=metricas_todas, mejor_modelo=mejor,
                real_test=real_test, media_obs=media_obs, predicciones=pred,
                modelos={"Random Forest": rf2, "XGBoost": xg2}, predictores=list(predictores),
                x_ajuste=x_aj, y_ajuste=y_aj, valores_imputacion=calcular_imputaciones(ajuste, predictores, metodo_na),
                dm=prueba_dm(real_test - pred["Random Forest"], real_test - pred["XGBoost"], perdida="cuadratica"),
                dm_abs=prueba_dm(real_test - pred["Random Forest"], real_test - pred["XGBoost"], perdida="absoluta"),
                origen_hp="tesis" if hp else "genéricos", fechas_test=test.get("Fecha"), guardias_test=test.get("Guardia"))

def particionar(df, usar_columna, temporal, prop_train, prop_valid):
    if usar_columna and "Particion" in df.columns:
        p = normalizar_particion(df["Particion"])
        if not pd.isna(p).any() and {"Entrenamiento", "Prueba"} <= set(p):
            return dict(train=df[p == "Entrenamiento"], valid=df[p == "Validación"], test=df[p == "Prueba"],
                        origen="columna Particion del archivo")
    n = len(df)
    if not temporal:
        df = df.iloc[np.random.default_rng(2026).permutation(n)]
    n_tr = max(1, int(np.floor(n * prop_train))); n_va = max(0, int(np.floor(n * prop_valid)))
    if n_tr + n_va >= n: n_va = max(0, n - n_tr - 1)
    return dict(train=df.iloc[:n_tr], valid=df.iloc[n_tr:n_tr + n_va], test=df.iloc[n_tr + n_va:],
                origen="partición temporal por proporciones" if temporal else "partición aleatoria")

# ---------------- Contraste de hipótesis (Tablas 8 y 9) ----------------
# Denominación Ho-Hi (general), Ho1-Hi1 (recuperación), Ho2-Hi2 (ley), Ho3-Hi3 (consumo), Ho4-Hi4 (RF vs XGBoost)
TITULO_TABLA8 = "Tabla 8. Contraste de las hipótesis Ho1-Hi1, Ho2-Hi2 y Ho3-Hi3 en el conjunto de prueba"
TITULO_TABLA9 = "Tabla 9. Contraste de la hipótesis Ho4-Hi4: Random Forest vs. XGBoost en el conjunto de prueba"
NOTA_TABLA8 = ("Nota. Cumple: R² ≥ 0,75, MAPE ≤ 10 % y MAE y RMSE menores que los del modelo base. Se acepta Hi1, Hi2 o Hi3\n"
               "si al menos un algoritmo cumple; si ninguno cumple, se acepta Ho. Ho-Hi: se acepta Hi si se aceptan Hi1, Hi2 y Hi3.")
NOTA_TABLA9 = ("Nota. Superioridad significativa: mayor R² y menor MAPE, MAE y RMSE que el otro algoritmo y Diebold-Mariano (HLN)\n"
               "con p < 0,05 con errores al cuadrado (e²) y absolutos (|e|). Se acepta Hi4 si ocurre en al menos una variable.")
COLS_T8 = ["Variable Objetivo", "Modelo", "Subconjunto", "n", "R²", "MAE", "RMSE", "MAPE (%)", "Cumple criterios", "Decisión"]
COLS_T9 = ["Variable Objetivo", "Modelo", "Subconjunto", "n", "R²", "MAE", "RMSE", "MAPE (%)", "p-valor", "Decisión"]
METRICAS_T8 = ["R²", "MAE", "RMSE", "MAPE (%)"]
METRICAS_T9 = ["R²", "MAE", "RMSE", "MAPE (%)", "p-valor"]
FUSION_T8 = ["Variable Objetivo", "Decisión"]              # celdas combinadas por variable objetivo
FUSION_T9 = ["Variable Objetivo", "p-valor", "Decisión"]

def numero_hipotesis(obj):
    return str(OBJETIVOS_TESIS.index(obj) + 1) if obj in OBJETIVOS_TESIS else None

def texto_decision(k, acepta):
    """k = "" -> Ho-Hi; "1".."4" -> específicas; None -> objetivo adicional (no es de la tesis)."""
    if k is None: return "Cumple los criterios" if acepta else "No cumple los criterios"
    return f"Se acepta Hi{k} y se rechaza Ho{k}" if acepta else f"Se acepta Ho{k} y se rechaza Hi{k}"

def fmt_p(p):
    return "—" if p is None or not np.isfinite(p) else ("< 0.001" if p < 0.001 else f"{p:.4f}")

def sintesis_hipotesis(res):
    t8, t9, acepta_obj, superior_obj = [], [], {}, {}
    objetivos = [o for o in OBJETIVOS_TESIS if o in res["objetivos"]] + [o for o in res["objetivos"] if o not in OBJETIVOS_TESIS]
    for obj in objetivos:
        r = res["resultados"][obj]; m = r["metricas"].set_index("Modelo"); k = numero_hipotesis(obj)
        n_test = len(r["real_test"])
        # Tabla 8: «Cumple criterios» por modelo y UNA decisión de Hi1/Hi2/Hi3 por variable
        cumplen = [mo for mo in ("Random Forest", "XGBoost") if m.loc[mo].Cumple_criterios == "Sí"]
        acepta = len(cumplen) > 0
        detalle = ("cumplen ambos algoritmos" if len(cumplen) == 2 else
                   (f"cumple {cumplen[0]}" if cumplen else "no cumple ningún algoritmo"))
        decision = f"{texto_decision(k, acepta)} ({detalle})"
        for mod in ("Modelo base", "Random Forest", "XGBoost"):
            f = m.loc[mod]; ok = None if mod == "Modelo base" else f.Cumple_criterios == "Sí"
            t8.append(dict(Objetivo=obj, Variable=etiqueta(obj), Modelo=mod, Subconjunto="Prueba", n=n_test,
                           R2=f.R2, MAE=f.MAE, RMSE=f.RMSE, MAPE=f.MAPE,
                           Cumple="Referencia" if ok is None else ("Cumple" if ok else "No cumple"),
                           Cumple_ok=ok, Decision=decision, Acepta=acepta))
        if k is not None: acepta_obj[obj] = acepta
        rf, xg = m.loc["Random Forest"], m.loc["XGBoost"]
        def domina(a, b):
            vals = [a.R2, a.MAPE, a.MAE, a.RMSE, b.R2, b.MAPE, b.MAE, b.RMSE]
            return all(np.isfinite(v) for v in vals) and a.R2 > b.R2 and a.MAPE < b.MAPE and a.MAE < b.MAE and a.RMSE < b.RMSE
        sup = "Random Forest" if domina(rf, xg) else ("XGBoost" if domina(xg, rf) else None)
        p_c, p_a = r["dm"]["p_valor"], r["dm_abs"]["p_valor"]
        ok4 = sup is not None and np.isfinite(p_c) and np.isfinite(p_a) and p_c < UMBRAL_P and p_a < UMBRAL_P
        dec4 = f"{sup} significativamente superior" if ok4 else "Sin superioridad significativa"
        if k is not None: superior_obj[obj] = ok4
        for mod, f in (("Random Forest", rf), ("XGBoost", xg)):
            t9.append(dict(Objetivo=obj, Variable=etiqueta(obj), Modelo=mod, Subconjunto="Prueba", n=n_test,
                           R2=f.R2, MAE=f.MAE, RMSE=f.RMSE, MAPE=f.MAPE, p_cuad=p_c, p_abs=p_a,
                           Superior=sup, Decision=dec4, Acepta=ok4))
    acepta_g = len(acepta_obj) > 0 and all(acepta_obj.values())
    acepta_4 = len(superior_obj) > 0 and any(superior_obj.values())     # Hi4: en al menos una variable
    return dict(tabla8=pd.DataFrame(t8), tabla9=pd.DataFrame(t9),
                general=dict(acepta=acepta_g, decision=texto_decision("", acepta_g), por_objetivo=acepta_obj),
                h4=dict(acepta=acepta_4, decision=texto_decision("4", acepta_4), por_objetivo=superior_obj))

def _f(x, d):
    return "—" if x is None or not np.isfinite(x) else f"{x:.{d}f}"

def tabla8_mostrar(t):
    out = pd.DataFrame({"Variable Objetivo": t.Variable, "Modelo": t.Modelo, "Subconjunto": t.Subconjunto,
                        "n": t.n.astype(str), "R²": [_f(x, 3) for x in t.R2], "MAE": [_f(x, 3) for x in t.MAE],
                        "RMSE": [_f(x, 3) for x in t.RMSE], "MAPE (%)": [_f(x, 2) for x in t.MAPE],
                        "Cumple criterios": t.Cumple, "Decisión": t.Decision})
    out["_grupo"] = t.Objetivo.values; out["_acepta"] = t.Acepta.values; out["_cumple"] = t.Cumple_ok.values
    return out

def tabla9_mostrar(t):
    out = pd.DataFrame({"Variable Objetivo": t.Variable, "Modelo": t.Modelo, "Subconjunto": t.Subconjunto,
                        "n": t.n.astype(str), "R²": [_f(x, 3) for x in t.R2], "MAE": [_f(x, 3) for x in t.MAE],
                        "RMSE": [_f(x, 3) for x in t.RMSE], "MAPE (%)": [_f(x, 2) for x in t.MAPE],
                        "p-valor": [f"e²: {fmt_p(a)}\n|e|: {fmt_p(b)}" for a, b in zip(t.p_cuad, t.p_abs)],
                        "Decisión": t.Decision})
    out["_grupo"] = t.Objetivo.values; out["_acepta"] = t.Acepta.values; out["_cumple"] = None
    return out

def tabla_hip_html(t, columnas, metricas, fusion):
    """Tabla HTML con encabezado en dos niveles y celdas combinadas por variable objetivo."""
    i_m = [columnas.index(c) for c in metricas]
    antes, despues = columnas[:min(i_m)], columnas[max(i_m) + 1:]
    cab1 = ("".join(f"<th rowspan='2'>{c}</th>" for c in antes) +
            f"<th colspan='{len(metricas)}'>Métricas de Evaluación de Modelo</th>" +
            "".join(f"<th rowspan='2'>{c}</th>" for c in despues))
    cab2 = "".join(f"<th>{c}</th>" for c in metricas)
    clase_dec = lambda a: "ref" if a is None or (isinstance(a, float) and np.isnan(a)) else ("ok" if a else "no")
    filas = []
    for g in dict.fromkeys(t["_grupo"]):
        sub = t[t["_grupo"] == g].reset_index(drop=True); n = len(sub)
        for i in range(n):
            celdas = []
            for c in columnas:
                clase = {"Variable Objetivo": "var", "p-valor": "pval"}.get(c, "")
                if c == "Decisión": clase = clase_dec(sub["_acepta"][i])
                if c == "Cumple criterios": clase = clase_dec(sub["_cumple"][i])
                if c in fusion:
                    if i == 0: celdas.append(f"<td rowspan='{n}' class='{clase}'>{sub[c][0]}</td>")
                else:
                    celdas.append(f"<td class='{clase}'>{sub[c][i]}</td>")
            filas.append("<tr>" + "".join(celdas) + "</tr>")
    return f"<table class='hip'><thead><tr>{cab1}</tr><tr>{cab2}</tr></thead><tbody>{''.join(filas)}</tbody></table>"

def html_hg(hg, titulo="Hipótesis general (Ho-Hi)",
            criterio="Criterios: R² ≥ 0,75 · MAPE ≤ 10 % · MAE y RMSE menores que los del modelo base."):
    a = hg["acepta"]
    return (f"<div class='hg' style='background:{'#F0FDF4' if a else '#FEF2F2'};border:1px solid {'#BBF7D0' if a else '#FECACA'}'>"
            f"<div class='lbl' style='color:{'#065F46' if a else '#991B1B'}'>{titulo}</div>"
            f"<div style='font-size:17px;font-weight:700'>{hg['decision']}</div>"
            f"<div class='det'>{criterio}</div></div>")

def html_tablas_hipotesis(res):
    s = res["sintesis"]
    return (f"<h3>{TITULO_TABLA8}</h3>" + tabla_hip_html(tabla8_mostrar(s["tabla8"]), COLS_T8, METRICAS_T8, FUSION_T8) +
            html_hg(s["general"]) + f"<div class='nota'>{NOTA_TABLA8.replace(chr(10), ' ')}</div>" +
            f"<h3>{TITULO_TABLA9}</h3>" + tabla_hip_html(tabla9_mostrar(s["tabla9"]), COLS_T9, METRICAS_T9, FUSION_T9) +
            html_hg(s["h4"], "Hipótesis específica 4 (Ho4-Hi4)",
                    "Se acepta Hi4 si hay superioridad significativa en al menos una variable.") +
            f"<div class='nota'>{NOTA_TABLA9.replace(chr(10), ' ')}</div>")

def ordenar_cronologicamente(df):
    if "Fecha" not in df.columns: return df
    fechas = pd.to_datetime(df["Fecha"], errors="coerce", dayfirst=not pd.api.types.is_datetime64_any_dtype(df["Fecha"]))
    if fechas.isna().all(): return df
    clave = pd.DataFrame({"f": fechas, "g": df["Guardia"].astype(str) if "Guardia" in df.columns else 0,
                          "i": np.arange(len(df))}, index=df.index)
    return df.loc[clave.sort_values(["f", "g", "i"], na_position="last").index]

def ejecutar_protocolo(df, objetivos, predictores, prop_train=0.7, prop_valid=0.15, temporal=True,
                       usar_columna=True, metodo_na="media", usar_hp_tesis=True):
    if not objetivos: raise ValueError("Selecciona al menos una variable objetivo.")
    if not predictores: raise ValueError("Selecciona al menos una variable predictora.")
    if prop_train + prop_valid >= 0.95: raise ValueError("La suma de proporciones debe ser menor que 0,95.")
    usar_columna = usar_columna and "Particion" in df.columns
    if not usar_columna: df = ordenar_cronologicamente(df)
    part = particionar(df, usar_columna, temporal, prop_train, prop_valid)
    if len(part["train"]) < 30 or len(part["test"]) < 5:
        raise ValueError("Se requieren ≥30 registros de entrenamiento y ≥5 de prueba.")
    cols_num = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    res = dict(objetivos=list(objetivos), resultados={}, particion=dict(
        origen=part["origen"], n_train=len(part["train"]), n_valid=len(part["valid"]), n_test=len(part["test"])))
    for i, obj in enumerate(objetivos, 1):
        if obj not in df.columns: raise ValueError(f"No se encuentra la variable objetivo: {obj}")
        preds = predictores_para(obj, predictores, cols_num)
        print(f"  [{i}/{len(objetivos)}] {etiqueta(obj)} · {len(preds)} predictores ...", end=" ")
        res["resultados"][obj] = entrenar_evaluar_target(obj, preds, part, usar_hp_tesis, metodo_na)
        print("listo")
    res["sintesis"] = sintesis_hipotesis(res)
    variables = list(dict.fromkeys(list(objetivos) + list(predictores)))
    res["descriptivos"] = descriptivos_por_subconjunto(part, variables)
    res["datos_particionados"] = pd.concat([d[variables].assign(Subconjunto=s)
                                            for s, d in zip(SUBCONJUNTOS, (part["train"], part["valid"], part["test"]))])
    res["metricas_todas"] = pd.concat([r["metricas_todas"].assign(Objetivo=etiqueta(o))
                                       for o, r in res["resultados"].items()], ignore_index=True)
    res["datos"] = df
    return res

In [ ]:
#@title 5 · Funciones de gráficos e importancia de variables
def importancia_rf_incmse(r):
    """%IncMSE como en randomForest de R: aumento del MSE fuera de bolsa al permutar cada variable,
    promediado entre árboles y dividido por su error estándar."""
    if "imp_rf" in r: return r["imp_rf"]
    modelo = r["modelos"]["Random Forest"]
    X = np.asarray(r["x_ajuste"], np.float32); y = np.asarray(r["y_ajuste"], float)
    n, p = X.shape; rng = np.random.default_rng(42)
    difs = np.full((len(modelo.estimators_), p), np.nan)
    for t, (arbol, en_bolsa) in enumerate(zip(modelo.estimators_, modelo.estimators_samples_)):
        oob = np.setdiff1d(np.arange(n), en_bolsa)
        if len(oob) < 2: continue
        Xo, yo = X[oob], y[oob]
        mse0 = np.mean((yo - arbol.predict(Xo, check_input=False)) ** 2)
        for j in range(p):
            Xp = Xo.copy(); Xp[:, j] = rng.permutation(Xp[:, j])
            difs[t, j] = np.mean((yo - arbol.predict(Xp, check_input=False)) ** 2) - mse0
    media, de = np.nanmean(difs, 0), np.nanstd(difs, 0, ddof=1)
    ntree = np.sum(np.isfinite(difs), 0)
    imp = np.where(de > 0, media / (de / np.sqrt(ntree)), 0.0)
    r["imp_rf"] = pd.Series(imp, index=r["predictores"])
    return r["imp_rf"]

def importancia_xgb_gain(r):
    if "imp_xgb" in r: return r["imp_xgb"]
    g = r["modelos"]["XGBoost"].get_score(importance_type="total_gain")
    s = pd.Series({r["predictores"][int(k[1:])]: v for k, v in g.items()})
    r["imp_xgb"] = (s / s.sum()).reindex(r["predictores"]).fillna(0.0)
    return r["imp_xgb"]

def _estilo(ax):
    for lado in ("top", "right"): ax.spines[lado].set_visible(False)
    ax.grid(axis="x", color="#E5E7EB", lw=0.8); ax.set_axisbelow(True)

def fig_metricas(res, figsize=(10, 4.8)):
    objs = sorted(res["objetivos"], key=lambda o: res["resultados"][o]["metricas"].query("Modelo != 'Modelo base'").R2.mean())
    fig, ax = plt.subplots(figsize=figsize)
    valores = []
    for i, obj in enumerate(objs):
        m = res["resultados"][obj]["metricas"].set_index("Modelo")
        for desp, mod in ((0.19, "Random Forest"), (-0.19, "XGBoost")):
            v = m.loc[mod, "R2"]; valores.append(v)
            ax.barh(i + desp, v, height=0.36, color=COLORES_MODELO[mod], label=mod if i == 0 else None)
            ax.text(v + (0.012 if v >= 0 else -0.012), i + desp, f"{v:.3f}", va="center",
                    ha="left" if v >= 0 else "right", fontsize=9, color="#374151")
    ax.axvline(UMBRAL_R2, ls="--", color="#B91C1C", lw=1.1)
    ax.text(UMBRAL_R2 + 0.01, -0.62, "umbral 0,75", color="#B91C1C", fontsize=8.5)
    ax.set_yticks(range(len(objs)), [etiqueta(o) for o in objs])
    ax.set_xlim(min(0, np.nanmin(valores) - 0.15), 1.12); ax.set_ylim(-0.75, len(objs) - 0.35)
    ax.set_xlabel("R²"); _estilo(ax); ax.grid(axis="y", visible=False)
    ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=2, frameon=False)
    fig.suptitle("Figura 1. Comparación de las métricas de desempeño en el conjunto de prueba",
                 x=0.01, ha="left", fontweight="bold", fontsize=12.5)
    fig.tight_layout()
    return fig

def dibujar_pred_real(ax, r, modelo=None, num_fig=None):
    modelo = modelo or r["mejor_modelo"]; et = etiqueta(r["objetivo"])
    real, pred = r["real_test"], r["predicciones"][modelo]
    m = r["metricas"].set_index("Modelo").loc[modelo]
    lo, hi = np.nanmin([real.min(), pred.min()]), np.nanmax([real.max(), pred.max()])
    marg = (hi - lo) * 0.04; lo, hi = lo - marg, hi + marg
    ax.plot([lo, hi], [lo, hi], ls="--", color="#9CA3AF", lw=1)
    ax.scatter(real, pred, s=18, color="#185FA5", alpha=0.75, edgecolor="none")
    ax.set_xlim(lo, hi); ax.set_ylim(lo, hi); ax.set_aspect("equal")
    ax.text(0.03, 0.97, f"R² = {m.R2:.3f}\nMAE = {m.MAE:.3f}\nMAPE = {m.MAPE:.2f} %", transform=ax.transAxes,
            va="top", fontsize=9.5, bbox=dict(fc="white", ec="#E5E7EB"))
    pre = f"Figura {num_fig}. " if num_fig else ""
    ax.set_title(f"{pre}Predicción vs. valor real: {et}\nModelo: {modelo} · prueba (n = {len(real)})",
                 fontsize=10.5, loc="left", fontweight="bold")
    ax.set_xlabel(f"Valor real"); ax.set_ylabel(f"Valor predicho")
    for lado in ("top", "right"): ax.spines[lado].set_visible(False)
    ax.grid(color="#EEF0F2", lw=0.8)

def dibujar_importancia(ax, r, tipo, top_n=10):
    imp = (importancia_rf_incmse(r) if tipo == "RF" else importancia_xgb_gain(r)).sort_values(ascending=False).head(top_n)[::-1]
    color = COLORES_MODELO["Random Forest" if tipo == "RF" else "XGBoost"]
    ax.barh([etiqueta(v) for v in imp.index], imp.values, color=color, height=0.68)
    rango = max(imp.max(), 1e-9)
    for i, v in enumerate(imp.values):
        ax.text(v + rango * 0.015, i, f"{v:.3f}", va="center", fontsize=8, color="#374151")
    ax.set_xlim(min(0, imp.min() * 1.1), rango * 1.22)
    ax.set_title("Random Forest (%IncMSE)" if tipo == "RF" else "XGBoost (Gain)", fontsize=10.5, loc="left", fontweight="bold")
    ax.tick_params(axis="y", labelsize=8.5); _estilo(ax); ax.grid(axis="y", visible=False)

def fig_importancia(r, num_fig=None):
    fig, axs = plt.subplots(1, 2, figsize=(12, 4.6))
    dibujar_importancia(axs[0], r, "RF"); dibujar_importancia(axs[1], r, "XGB")
    pre = f"Figura {num_fig}. " if num_fig else ""
    fig.suptitle(f"{pre}Importancia de variables: {etiqueta(r['objetivo'])}", x=0.01, ha="left", fontweight="bold")
    fig.tight_layout(); return fig

def fig_pred_real(r, num_fig=None):
    fig, ax = plt.subplots(figsize=(6.2, 5.6)); dibujar_pred_real(ax, r, num_fig=num_fig); fig.tight_layout(); return fig

def dibujar_boxplot(ax, dp, var):
    datos = [pd.to_numeric(dp.loc[dp.Subconjunto == s, var], errors="coerce").dropna().to_numpy() for s in SUBCONJUNTOS]
    cajas = ax.boxplot(datos, patch_artist=True, widths=0.55, medianprops=dict(color="#111827"),
                       flierprops=dict(marker="o", markersize=2.5, alpha=0.5))
    for caja, s in zip(cajas["boxes"], SUBCONJUNTOS): caja.set_facecolor(COLORES_SUBCONJUNTO[s]); caja.set_alpha(0.85)
    rng = np.random.default_rng(1)
    for i, z in enumerate(datos, 1):
        ax.scatter(i + rng.uniform(-0.15, 0.15, len(z)), z, s=4, color="#111827", alpha=0.2)
    ax.set_xticks([1, 2, 3], SUBCONJUNTOS); ax.set_title(etiqueta(var), fontsize=10.5, loc="left")
    for lado in ("top", "right"): ax.spines[lado].set_visible(False)
    ax.grid(axis="y", color="#EEF0F2")

In [ ]:
#@title 6 · Vistas: Portada, Datos y Resultados
CSS = """<style>
.flt{font-family:Inter,'Segoe UI',Roboto,Arial,sans-serif;color:#111827;background:#fff;padding:14px;border-radius:10px;max-width:1180px}
.flt .hero{background:linear-gradient(135deg,#0F6E56,#0B4F3E);color:#fff;border-radius:14px;padding:22px 28px;margin-bottom:16px}
.flt .hero .sup{font-size:12px;opacity:.85}.flt .hero h1{font-size:22px;margin:6px 0 10px;color:#fff;line-height:1.25}
.flt .hero p{margin:2px 0;font-size:13px;color:#fff;opacity:.95}
.flt .grid{display:grid;grid-template-columns:repeat(auto-fit,minmax(230px,1fr));gap:12px;margin-bottom:16px}
.flt .card{border:1px solid #E5E7EB;border-radius:12px;padding:14px 16px;background:#fff}
.flt .lbl{font-size:11px;letter-spacing:.06em;text-transform:uppercase;color:#6B7280}
.flt .val{font-size:24px;font-weight:700;margin:4px 0;color:#111827}.flt .det{font-size:12px;color:#6B7280}
.flt .proto{background:#F8FAFC;border:1px solid #E5E7EB;border-radius:12px;padding:10px 18px;font-size:13px;margin-bottom:16px}
.flt .hg{border-radius:12px;padding:12px 18px;margin-bottom:16px}
.flt .badge{display:inline-block;padding:2px 10px;border-radius:999px;font-size:11px;font-weight:700}
.flt .met{display:inline-block;margin:8px 18px 0 0}.flt .met b{font-size:19px;display:block}
.flt table.t{border-collapse:collapse;font-size:12.5px;margin:6px 0 14px}
.flt table.t th{background:#0F6E56;color:#fff;padding:6px 10px;text-align:left;font-weight:600}
.flt table.t td{padding:5px 10px;border-bottom:1px solid #E5E7EB;color:#111827}
.flt h3{color:#0F6E56;margin:18px 0 6px;font-size:16px}.flt .nota{font-size:12px;color:#6B7280}
.flt .card-seccion{border:1px solid #E5E7EB;border-radius:14px;padding:20px 24px;margin-bottom:16px;background:#fff}
.flt .card-seccion h4{margin-top:0;color:#0F6E56;font-size:19px}.flt .card-seccion h5{color:#111827;font-size:15px;margin:16px 0 6px}
.flt .card-seccion p,.flt .card-seccion li{color:#374151;font-size:13.5px;line-height:1.6}
.flt table.hip{border-collapse:collapse;font-size:12.5px;margin:6px 0 8px;width:100%}
.flt table.hip th{background:#0F6E56;color:#fff;padding:6px 8px;text-align:center;font-weight:600;border:1px solid #D1D5DB}
.flt table.hip td{padding:6px 8px;border:1px solid #D1D5DB;text-align:center;vertical-align:middle;color:#111827}
.flt table.hip td.var{font-weight:600;background:#F9FAFB}.flt table.hip td.pval{white-space:pre}
.flt table.hip td.ok{background:#E1F5EE;color:#065F46;font-weight:600}
.flt table.hip td.no{background:#FEE2E2;color:#991B1B;font-weight:600}
.flt table.hip td.ref{background:#F3F4F6;color:#6B7280}
</style>"""

def fmt(x, d=3):
    return "—" if x is None or (isinstance(x, float) and not np.isfinite(x)) else (f"{x:.{d}f}" if isinstance(x, (float, np.floating)) else str(x))

def tabla_html(df, colores=None, decimales=4):
    cab = "".join(f"<th>{c}</th>" for c in df.columns)
    filas = []
    for i, (_, fila) in enumerate(df.iterrows()):
        fondo = colores[i] if colores is not None else ("#FFFFFF" if i % 2 == 0 else "#F9FAFB")
        filas.append(f"<tr style='background:{fondo}'>" + "".join(f"<td>{fmt(v, decimales)}</td>" for v in fila.values) + "</tr>")
    return f"<table class='t'><thead><tr>{cab}</tr></thead><tbody>{''.join(filas)}</tbody></table>"

def mostrar(html):
    display(HTML(CSS + f"<div class='flt'>{html}</div>"))

def texto_protocolo(res):
    p = res["particion"]
    return [f"Partición: {p['origen']} — entrenamiento = {p['n_train']}, validación = {p['n_valid']}, prueba = {p['n_test']} registros.",
            "Modelo base: media histórica del conjunto de ajuste (entrenamiento + validación).",
            "Entrenamiento y validación: modelos ajustados solo con entrenamiento.",
            "Prueba: modelos reentrenados con entrenamiento + validación, evaluados una única vez.",
            "Criterios Ho-Hi y Ho1-Hi1 a Ho3-Hi3: R² ≥ 0,75; MAPE ≤ 10 %; MAE y RMSE menores que los del modelo base.",
            "Ho4-Hi4: mayor R² y menor MAPE, MAE y RMSE que el otro modelo, y DM con p < 0,05 con errores al cuadrado y absolutos.",
            "Las dosificaciones de los cinco reactivos se usan como predictores de las tres variables; "
            "el consumo específico total se usa como predictor de la recuperación y de la ley."]

def html_portada(res):
    hero = ("<div class='hero'><div class='sup'>Trabajo de investigación para obtener el grado de maestro en Inteligencia Artificial</div>"
            "<h1>Aplicación de machine learning para predecir la flotación de relaves de lixiviación, Lima 2026</h1>"
            "<p>Barboza Duran, Edwin Miguel · Ruiz Torres, Alfonso</p><p>Asesor: MSc. Ruben Quispe Llacctarimay</p>"
            "<p style='font-size:12px'>Universidad César Vallejo · Escuela de Posgrado · Maestría en Inteligencia Artificial · ODS 9</p></div>")
    kpis, tarjetas = [], []
    for obj in res["objetivos"]:
        r = res["resultados"][obj]; m = r["metricas"].set_index("Modelo").loc[r["mejor_modelo"]]
        kpis.append(f"<div class='card'><div class='lbl'>{etiqueta(obj)}</div><div class='val'>R² = {m.R2:.3f}</div>"
                    f"<div class='det'>{r['mejor_modelo']} · MAPE = {m.MAPE:.2f} %</div></div>")
        apto = m.Cumple_criterios == "Sí"; c = "#0F6E56" if apto else "#B91C1C"
        col = lambda ok: "#0F6E56" if ok else "#B91C1C"
        mets = [("R²", f"{m.R2:.3f}", m.R2 >= UMBRAL_R2), ("MAPE", f"{m.MAPE:.2f} %", m.MAPE <= UMBRAL_MAPE),
                ("Red. MAE", f"{m.Red_MAE_vs_base:.0f} %", m.Red_MAE_vs_base > 0),
                ("Red. RMSE", f"{m.Red_RMSE_vs_base:.0f} %", m.Red_RMSE_vs_base > 0)]
        tarjetas.append(f"<div class='card' style='border-left:5px solid {c}'><div class='lbl'>{etiqueta(obj)}</div>"
                        f"<div class='val'>{r['mejor_modelo']}</div><span class='badge' style='background:{'#D1FAE5' if apto else '#FEE2E2'};color:{c}'>"
                        f"{'APTO' if apto else 'NO APTO'}</span><div>" +
                        "".join(f"<span class='met'><b style='color:{col(ok)}'>{v}</b><span class='lbl'>{n}</span></span>" for n, v, ok in mets) +
                        "</div></div>")
    hg_html = html_hg(res["sintesis"]["general"])
    proto = "<div class='proto'><b>Protocolo de la tesis</b><ul>" + "".join(f"<li>{t}</li>" for t in texto_protocolo(res)) + "</ul></div>"
    return hero + f"<div class='grid'>{''.join(kpis)}</div>" + proto + hg_html + f"<div class='grid'>{''.join(tarjetas)}</div>"

def tabla4_df(res):
    filas, colores = [], []
    for obj in res["objetivos"]:
        for _, m in res["resultados"][obj]["metricas"].iterrows():
            base = m.Modelo == "Modelo base"
            filas.append({"Variable objetivo": etiqueta(obj), "Modelo": m.Modelo, "R²": m.R2, "MAE": m.MAE, "RMSE": m.RMSE,
                          "MAPE (%)": m.MAPE, "Red. MAE (%)": "—" if base else m.Red_MAE_vs_base,
                          "Red. RMSE (%)": "—" if base else m.Red_RMSE_vs_base, "Cumple criterios": m.Cumple_criterios})
            colores.append("#D7F0E6" if m.EsMejor else "#FFFFFF")
    return pd.DataFrame(filas), colores

def tabla_subconjunto_df(r):
    t = r["metricas_todas"].copy()
    return t[["Modelo", "Subconjunto", "n", "R2", "MAE", "RMSE", "MAPE", "RRMSE"]].rename(
        columns={"R2": "R²", "MAPE": "MAPE (%)", "RRMSE": "RRMSE (%)"})

import base64, uuid
def img_html(fig, ancho="100%", dpi=110):
    """Convierte una figura de matplotlib en una imagen incrustada (no requiere widgets)."""
    buf = io.BytesIO(); fig.savefig(buf, format="png", dpi=dpi, bbox_inches="tight"); plt.close(fig)
    return f"<img style='max-width:{ancho};height:auto' src='data:image/png;base64,{base64.b64encode(buf.getvalue()).decode()}'/>"

def html_datos(res):
    p = res["particion"]
    tarjetas = "".join(f"<div class='card'><div class='lbl'>{n}</div><div class='val'>{v}</div><div class='det'>registros</div></div>"
                       for n, v in (("Entrenamiento", p["n_train"]), ("Validación", p["n_valid"]), ("Prueba", p["n_test"])))
    html = (f"<h3>Partición</h3><div class='grid'>{tarjetas}<div class='card'><div class='lbl'>Origen</div>"
            f"<div class='det' style='margin-top:8px'>{p['origen']}</div></div></div>")
    variables = list(dict.fromkeys(res["descriptivos"].Variable))
    opciones = "".join(f"<option value='{v}'>{etiqueta(v)}</option>" for v in variables)
    html += ("<h3>Estadísticos descriptivos por subconjunto</h3><label class='det'>Variable: </label>"
             "<select style='padding:4px 8px;font-size:13px;min-width:380px' onchange=\"var r=this.closest('.pane');"
             "r.querySelectorAll('.var-blk').forEach(function(e){e.style.display=(e.dataset.v===this.value)?'block':'none';},this);\">"
             f"{opciones}</select>")
    for i, v in enumerate(variables):
        d = res["descriptivos"][res["descriptivos"].Variable == v].drop(columns="Variable")
        fig, ax = plt.subplots(figsize=(10, 3.4)); dibujar_boxplot(ax, res["datos_particionados"], v); fig.tight_layout()
        html += (f"<div class='var-blk' data-v='{v}' style='display:{'block' if i == 0 else 'none'}'>"
                 + tabla_html(d.rename(columns={"n_NA": "n NA", "CV_pct": "CV (%)"}), decimales=3) + img_html(fig, dpi=80) + "</div>")
    datos = res["datos"].head(15).copy()
    for c in datos.columns:
        if pd.api.types.is_datetime64_any_dtype(datos[c]): datos[c] = datos[c].dt.strftime("%Y-%m-%d")
    datos = datos.rename(columns=lambda c: etiqueta(c))
    html += (f"<h3>Dataset cargado (primeras 15 de {len(res['datos'])} filas)</h3>"
             "<div style='overflow-x:auto'>" + tabla_html(datos, decimales=3) + "</div>")
    return html

def html_resultados(res):
    t4, c4 = tabla4_df(res)
    html = ("<h3>Tabla 4. Desempeño predictivo de los modelos en el conjunto de prueba</h3>" + tabla_html(t4, c4) +
            "<div class='nota'>Fila verde = mejor modelo. Reducciones respecto al modelo base. "
            "Cumple: R² ≥ 0,75; MAPE ≤ 10 %; MAE y RMSE menores que los del modelo base.</div>" + img_html(fig_metricas(res), "900px"))
    for obj in res["objetivos"]:
        r = res["resultados"][obj]; num = NUMERACION.get(obj, {})
        html += (f"<h3>Tabla {num.get('tabla', '')}. {etiqueta(obj)} por subconjunto</h3>" + tabla_html(tabla_subconjunto_df(r)) +
                 f"<div class='nota'>Mejor modelo: {r['mejor_modelo']} · hiperparámetros {r['origen_hp']} · "
                 f"{len(r['predictores'])} predictores</div>"
                 + img_html(fig_pred_real(r, num.get("pred")), "560px") + img_html(fig_importancia(r, num.get("imp")), "1050px"))
    html += html_tablas_hipotesis(res)
    return html

# Texto fijo del informe (idéntico a las pestañas Discusión y Conclusiones de la app R v7).
# No se recalcula si cambian los datos.
HTML_DISCUSION = """<div class="card-seccion">
  <h4>IV. Discusión de resultados</h4>
  <p>Cuatro condiciones permiten leer los resultados sin sobreestimarlos ni subestimarlos:</p>
  <h5>1. El protocolo de evaluación no favoreció el resultado</h5>
  <p>La partición fue temporal: los 83 registros de prueba son los más recientes y no intervinieron en ninguna etapa del ajuste. Las particiones aleatorias en datos con dependencia temporal subestiman el error de predicción (Hewamalage et al., 2023; Lones, 2024).</p>
  <h5>2. Los modelos siguieron un cambio de nivel de operación</h5>
  <p>El nivel medio de la recuperación se desplazó 1.06 desviaciones estándar y el del consumo 2.09 desviaciones estándar respecto a la media histórica. Los valores de consumo del periodo de prueba (0.7 a 3.0 kg/t) fueron inferiores al promedio trimestral de 3.98 kg/t, consistente con una reducción de la dosificación hacia el final del periodo (Lu et al., 2019).</p>
  <h5>3. Random Forest vs. XGBoost: la estructura del problema decide</h5>
  <p>XGBoost ganó en consumo y ley; Random Forest en recuperación. La ventaja de XGBoost en el consumo se explica porque el boosting aproxima con eficiencia relaciones aditivas, como la suma de dosificaciones de la ecuación i3. La de Random Forest en recuperación se asocia con la reducción de varianza del promedio de árboles decorrelacionados cuando la respuesta contiene ruido de medición propio del balance metalúrgico.</p>
  <h5>4. La interpretabilidad valida la coherencia metalúrgica</h5>
  <p>En las tres variables, las dos medidas de importancia (permutación en RF, ganancia en XGBoost) coincidieron en las variables principales: el activador de la pirita, la concentración de sólidos y la composición en metales base para la recuperación; la ley de Fe, la humedad y la ley de As para la ley; y las propias dosificaciones para el consumo.</p>
  <h5>Limitaciones del estudio</h5>
  <ol>
    <li>Los datos provienen de una sola planta y de un periodo de nueve meses.</li>
    <li>El conjunto de prueba (n = 83) abarcó un único periodo de operación.</li>
    <li>No se documentó la disponibilidad temporal de las determinaciones del bloque ccPy.</li>
    <li>Las medidas de importancia no indican el sentido del efecto.</li>
    <li>No se estimaron intervalos de predicción que comuniquen la incertidumbre.</li>
  </ol>
</div>"""
HTML_CONCLUSIONES = """<div class="card-seccion">
  <h4>V. Conclusiones</h4>
  <ol>
    <li>
      <b>General.</b>
       El modelo predictivo basado en IA (Random Forest y XGBoost), entrenado con 547 registros operacionales y 38 predictores, permitió predecir la flotación de relaves de lixiviación con desempeño adecuado; se aceptó la Hi y se rechazó la Ho. R² de prueba: 0.972 (recuperación), 0.989 (ley), 0.885 (consumo); MAPE de 0.96 %, 0.38 % y 9.29 %; reducciones del MAE y del RMSE superiores al 85 %.
    </li>
    <li>
      <b>Ho1-Hi1.</b>
       Random Forest obtuvo R² = 0.972, MAE = 0.695 pp, RMSE = 0.850 pp y MAPE = 0.96 %, con reducciones de 87.3 % y 88.5 % frente al modelo base. XGBoost alcanzó R² = 0.965. Se aceptó la Hi1 y se rechazó la Ho1.
    </li>
    <li>
      <b>Ho2-Hi2.</b>
       XGBoost obtuvo R² = 0.989, MAE = 0.029 g/t, RMSE = 0.037 g/t y MAPE = 0.38 %, con reducciones de 89.9 % y 89.4 %. Random Forest alcanzó R² = 0.985. Se aceptó la Hi2 y se rechazó la Ho2.
    </li>
    <li>
      <b>Ho3-Hi3.</b>
       XGBoost obtuvo R² = 0.885, MAE = 0.146 kg/t, RMSE = 0.186 kg/t y MAPE = 9.29 %, con reducciones de 87.2 % y 85.3 %. Random Forest no cumplió los criterios (R² = 0.740; MAPE = 14.70 %). Se aceptó la Hi3 y se rechazó la Ho3 solo con XGBoost.
    </li>
    <li>
      <b>Ho4-Hi4.</b>
       Ningún algoritmo fue superior en las tres variables. Random Forest superó a XGBoost en la recuperación (mayor R² y menores MAPE, MAE y RMSE; Diebold-Mariano con p = 0.007 con errores al cuadrado y p = 0.014 con errores absolutos) y XGBoost superó a Random Forest en el consumo (p &lt; 0.001 con ambas funciones de pérdida); en la ley, Random Forest obtuvo mejores métricas y p = 0.033 con errores al cuadrado, pero p = 0.080 con errores absolutos, por lo que no hubo superioridad significativa. Como la superioridad significativa se presentó en al menos una variable, se aceptó la Hi4 y se rechazó la Ho4.
    </li>
  </ol>
</div>
<div class="card-seccion">
  <h4>VI. Recomendaciones</h4>
  <ol>
    <li>
      <b>A la gerencia de operaciones.</b>
       Implementar los modelos como sensores blandos de apoyo a la decisión (XGBoost para ley y consumo; Random Forest o XGBoost para recuperación) durante un periodo de validación operativa en paralelo con los ensayos de laboratorio.
    </li>
    <li>
      <b>Al área de control de procesos.</b>
       Establecer un esquema de reentrenamiento periódico de origen móvil (mensual o ante cambio de régimen) y un monitoreo de deriva.
    </li>
    <li>
      <b>Al laboratorio metalúrgico.</b>
       Documentar la técnica analítica y el momento en que queda disponible cada determinación del bloque ccPy respecto a la ley de Au.
    </li>
    <li>
      <b>Al área de metalurgia.</b>
       Complementar la importancia de variables con valores SHAP y gráficos de dependencia parcial, y validar los efectos con pruebas controladas en planta.
    </li>
    <li>
      <b>A futuros investigadores.</b>
       Ampliar la base a uno o más años de operación y aplicar validación temporal de múltiples orígenes; comparar con CatBoost y modelos informados por física; estimar intervalos de predicción; incorporar caracterización mineralógica y de liberación del oro.
    </li>
    <li>
      <b>A la Escuela de Posgrado.</b>
       Promover convenios con plantas concentradoras para acceder a bases de mayor extensión y frecuencia.
    </li>
  </ol>
</div>"""

def mostrar_panel(res):
    """Pestañas Portada, Datos, Resultados, Discusión y Conclusiones (como la app Shiny), solo con HTML y CSS."""
    u = uuid.uuid4().hex[:6]
    nombres = ("Portada", "Datos", "Resultados", "Discusión", "Conclusiones")
    contenidos = (html_portada(res), html_datos(res), html_resultados(res), HTML_DISCUSION, HTML_CONCLUSIONES)
    css = (f"<style>.tabs{u}>input{{display:none}}.tabs{u}>label{{display:inline-block;padding:8px 20px;margin-right:4px;"
           f"border:1px solid #E5E7EB;border-bottom:none;border-radius:8px 8px 0 0;cursor:pointer;background:#F9FAFB;"
           f"color:#374151;font-weight:600;font-size:14px}}.tabs{u}>input:checked+label{{background:#fff;color:#0F6E56;"
           f"border-color:#0F6E56}}.tabs{u}>.pane{{display:none;border-top:2px solid #0F6E56;padding-top:14px}}"
           + ",".join(f"#t{i}{u}:checked~.p{i}" for i in range(len(nombres))) + "{display:block}</style>")
    cab = "".join(f"<input type='radio' name='g{u}' id='t{i}{u}' {'checked' if i == 0 else ''}>"
                  f"<label for='t{i}{u}'>{n}</label>" for i, n in enumerate(nombres))
    paneles = "".join(f"<div class='pane p{i}'>{c}</div>" for i, c in enumerate(contenidos))
    display(HTML(CSS + css + f"<div class='flt'><div class='tabs{u}'>{cab}{paneles}</div></div>"))

In [ ]:
#@title 7 · Informe PDF (Portada, Datos y Resultados)
class InformePDF:
    """Página horizontal de 11 × 8,5 pulgadas; coloca bloques de arriba hacia abajo y pagina sola."""
    ANCHO, ALTO, MARGEN = 11.0, 8.5, 0.5
    def __init__(self, ruta):
        self.pdf = PdfPages(ruta); self.fig = None; self.y = 0
    def _nueva(self):
        self._cerrar(); self.fig = plt.figure(figsize=(self.ANCHO, self.ALTO)); self.y = self.ALTO - self.MARGEN
    def _cerrar(self):
        if self.fig is not None:
            self.pdf.savefig(self.fig); plt.close(self.fig); self.fig = None
    def asegurar(self, alto):
        if self.fig is None or self.y - alto < self.MARGEN: self._nueva()
    def salto(self): self._nueva()
    def texto(self, t, tam=10, negrita=False, color="#111827", alto=None):
        alto = alto or tam / 72 * 1.9
        self.asegurar(alto)
        self.fig.text(self.MARGEN / self.ANCHO, (self.y - alto * 0.7) / self.ALTO, t, fontsize=tam,
                      fontweight="bold" if negrita else "normal", color=color)
        self.y -= alto
    @staticmethod
    def _formatear(df, decimales):
        return [[fmt(v, decimales) for v in fila] for fila in df.itertuples(index=False)]
    def tabla(self, df, colores=None, tam=8, decimales=3, titulo=None, nota=None):
        celdas = self._formatear(df, decimales)
        cabecera = [textwrap.fill(str(c), 14) for c in df.columns]
        largos = [max([len(str(h).split("\n")[0]) for h in [cabecera[j]]] + [len(f[j]) for f in celdas]) for j in range(len(cabecera))]
        ancho = min(self.ANCHO - 2 * self.MARGEN, sum(l * tam * 0.0078 + 0.22 for l in largos))
        lineas_cab = max(h.count("\n") + 1 for h in cabecera)
        alto_fila = tam / 72 * 2.0
        alto = alto_fila * (len(celdas) + lineas_cab * 0.75 + 0.25)
        extra = (0.32 if titulo else 0) + (0.3 if nota else 0)
        self.asegurar(alto + extra)
        if titulo: self.texto(titulo, 11, True, alto=0.32)
        x0 = (self.ANCHO - ancho) / 2
        ax = self.fig.add_axes([x0 / self.ANCHO, (self.y - alto) / self.ALTO, ancho / self.ANCHO, alto / self.ALTO]); ax.axis("off")
        t = ax.table(cellText=celdas, colLabels=cabecera, cellLoc="center", bbox=[0, 0, 1, 1],
                     colWidths=[l / sum(largos) for l in largos])
        t.auto_set_font_size(False); t.set_fontsize(tam)
        for (i, j), celda in t.get_celld().items():
            celda.set_edgecolor("#D1D5DB"); celda.set_linewidth(0.5)
            if i == 0:
                celda.set_facecolor(VERDE); celda.get_text().set_color("white"); celda.get_text().set_fontweight("bold")
                celda.set_height(celda.get_height() * lineas_cab * 0.75)
            else:
                celda.set_facecolor(colores[i - 1] if colores is not None else ("#FFFFFF" if i % 2 else "#F9FAFB"))
        self.y -= alto + 0.12
        if nota: self.texto(nota, 7.5, color="#4B5563", alto=0.28)
    def tabla_hip(self, t, columnas, metricas, fusion, anchos, titulo=None, tam=8):
        """Tablas 8 y 9: encabezado en dos niveles y celdas combinadas (anchos en pulgadas)."""
        i_m = [columnas.index(c) for c in metricas]
        esc = (self.ANCHO - 2 * self.MARGEN) / sum(anchos) if sum(anchos) > self.ANCHO - 2 * self.MARGEN else 1.0
        anchos = [a * esc for a in anchos]
        def envolver(x, ancho):
            car = max(5, int(ancho / (tam * 0.0078)))
            return "\n".join(textwrap.fill(p, car) for p in str(x).split("\n"))
        txt = t.copy()
        for c, a in zip(columnas, anchos): txt[c] = [envolver(v, a) for v in t[c]]
        lin = lambda s: s.count("\n") + 1
        al = tam / 72 * 1.35; mg = 0.12
        alturas = [max(lin(txt[c].iloc[i]) for c in columnas if c not in fusion) * al + mg for i in range(len(txt))]
        for g in dict.fromkeys(txt["_grupo"]):
            idx = [i for i in range(len(txt)) if txt["_grupo"].iloc[i] == g]
            nec = max(lin(txt[c].iloc[idx[0]]) for c in fusion) * al + mg
            tot = sum(alturas[i] for i in idx)
            if tot < nec:
                for i in idx: alturas[i] *= nec / tot
        h_cab = 0.26
        alto = 2 * h_cab + sum(alturas)
        self.asegurar(alto + (0.32 if titulo else 0) + 0.1)
        if titulo: self.texto(titulo, 11, True, alto=0.32)
        x0 = (self.ANCHO - sum(anchos)) / 2
        xs = [x0 + sum(anchos[:j]) for j in range(len(anchos) + 1)]
        f = self.fig; y_top = self.y
        def celda(x1, x2, y1, y2, texto, fondo, color="#111827", negrita=False):
            f.add_artist(Rectangle((x1 / self.ANCHO, y2 / self.ALTO), (x2 - x1) / self.ANCHO, (y1 - y2) / self.ALTO,
                                   transform=f.transFigure, facecolor=fondo, edgecolor="#B8BEC6", linewidth=0.5))
            f.text((x1 + x2) / 2 / self.ANCHO, (y1 + y2) / 2 / self.ALTO, texto, ha="center", va="center",
                   fontsize=tam, color=color, fontweight="bold" if negrita else "normal", linespacing=1.15)
        for j, c in enumerate(columnas):
            if j in i_m: celda(xs[j], xs[j + 1], y_top - h_cab, y_top - 2 * h_cab, c, VERDE, "white", True)
            else: celda(xs[j], xs[j + 1], y_top, y_top - 2 * h_cab, c, VERDE, "white", True)
        celda(xs[min(i_m)], xs[max(i_m) + 1], y_top, y_top - h_cab, "Métricas de Evaluación de Modelo", VERDE, "white", True)
        ys = [y_top - 2 * h_cab]
        for a in alturas: ys.append(ys[-1] - a)
        def colores_dec(a):
            if a is None or (isinstance(a, float) and np.isnan(a)): return "#F3F4F6", "#6B7280"
            return ("#E1F5EE", "#065F46") if a else ("#FEE2E2", "#991B1B")
        for g in dict.fromkeys(txt["_grupo"]):
            idx = [i for i in range(len(txt)) if txt["_grupo"].iloc[i] == g]
            for i in idx:
                for j, c in enumerate(columnas):
                    es_dec = c in ("Decisión", "Cumple criterios")
                    estado = txt["_cumple"].iloc[i] if c == "Cumple criterios" else txt["_acepta"].iloc[i]
                    fondo, col = colores_dec(estado) if es_dec else ("#F9FAFB" if c == "Variable Objetivo" else "white", "#111827")
                    if c in fusion:
                        if i == idx[0]:
                            celda(xs[j], xs[j + 1], ys[idx[0]], ys[idx[-1] + 1], txt[c].iloc[i], fondo, col,
                                  es_dec or c == "Variable Objetivo")
                    else:
                        celda(xs[j], xs[j + 1], ys[i], ys[i + 1], txt[c].iloc[i], fondo, col, es_dec)
        self.y = ys[-1] - 0.1

    def figura(self, fig):
        self._cerrar(); fig.set_size_inches(self.ANCHO, self.ALTO); self.pdf.savefig(fig); plt.close(fig)
    def cerrar(self):
        self._cerrar(); self.pdf.close()

def generar_informe_pdf(res, ruta):
    inf = InformePDF(ruta)
    # ---------------- 1. Portada ----------------
    inf.salto(); f = inf.fig
    f.add_artist(Rectangle((0, 1 - 1.75 / 8.5), 1, 1.75 / 8.5, transform=f.transFigure, color=VERDE, zorder=0))
    lineas = [("Trabajo de investigación para obtener el grado de maestro en Inteligencia Artificial", 9, "normal"),
              ("Aplicación de machine learning para predecir la flotación de relaves de lixiviación, Lima 2026", 15, "bold"),
              ("Barboza Duran, Edwin Miguel · Ruiz Torres, Alfonso", 10, "normal"),
              ("Asesor: MSc. Ruben Quispe Llacctarimay", 10, "normal"),
              ("Universidad César Vallejo · Escuela de Posgrado · Maestría en Inteligencia Artificial · ODS 9", 8.5, "normal"),
              (f"Informe de comparación generado el {datetime.now():%d/%m/%Y %H:%M}", 8, "normal")]
    y = 8.5 - 0.35
    for t, tam, peso in lineas:
        f.text(0.045, y / 8.5, t, color="white", fontsize=tam, fontweight=peso, va="top"); y -= tam / 72 * 1.75 + 0.05
    hg = res["sintesis"]["general"]; acepta = hg["acepta"]
    f.add_artist(Rectangle((0.045, (8.5 - 2.65) / 8.5), 0.91, 0.62 / 8.5, transform=f.transFigure,
                               facecolor="#F0FDF4" if acepta else "#FEF2F2", edgecolor="#BBF7D0" if acepta else "#FECACA"))
    f.text(0.06, (8.5 - 2.2) / 8.5, "HIPÓTESIS GENERAL (Ho-Hi)", fontsize=8.5, fontweight="bold", color="#065F46" if acepta else "#991B1B")
    f.text(0.06, (8.5 - 2.5) / 8.5, hg["decision"], fontsize=13, fontweight="bold")
    inf.y = 8.5 - 2.9
    kp = []
    for obj in res["objetivos"]:
        r = res["resultados"][obj]; m = r["metricas"].set_index("Modelo").loc[r["mejor_modelo"]]
        kp.append({"Variable objetivo": etiqueta(obj), "Mejor modelo": r["mejor_modelo"],
                   "Veredicto": "APTO" if m.Cumple_criterios == "Sí" else "NO APTO", "R²": f"{m.R2:.3f}",
                   "MAPE (%)": f"{m.MAPE:.2f}", "Red. MAE (%)": f"{m.Red_MAE_vs_base:.1f}", "Red. RMSE (%)": f"{m.Red_RMSE_vs_base:.1f}"})
    kp = pd.DataFrame(kp)
    inf.tabla(kp, ["#E1F5EE" if v == "APTO" else "#FEE2E2" for v in kp.Veredicto], tam=10,
              titulo="Indicadores del mejor modelo en el conjunto de prueba")
    inf.texto("Protocolo de la tesis", 11, True, alto=0.34)
    for t in texto_protocolo(res):
        inf.texto("•  " + t, 8.8, color="#374151", alto=0.24)

    # ---------------- 2. Datos ----------------
    inf.salto(); inf.texto("2. Datos", 16, True, VERDE, alto=0.42)
    p = res["particion"]
    inf.tabla(pd.DataFrame({"Subconjunto": SUBCONJUNTOS, "Registros": [p["n_train"], p["n_valid"], p["n_test"]],
                            "Origen": [p["origen"]] * 3}), tam=9.5, titulo="Tamaño de los subconjuntos")
    d = res["descriptivos"]; d_obj = d[d.Variable.isin(res["objetivos"])].copy()
    d_obj["Variable"] = d_obj.Variable.map(etiqueta)
    inf.tabla(d_obj.rename(columns={"n_NA": "n NA", "CV_pct": "CV (%)"}), tam=8.5, decimales=3,
              titulo="Estadísticos descriptivos de las variables objetivo por subconjunto")
    fig, axs = plt.subplots(1, len(res["objetivos"]), figsize=(11, 8.5))
    for ax, obj in zip(np.atleast_1d(axs), res["objetivos"]): dibujar_boxplot(ax, res["datos_particionados"], obj)
    fig.suptitle("Distribución de las variables objetivo por subconjunto", fontsize=13, fontweight="bold", color=VERDE)
    fig.tight_layout(rect=(0.02, 0.03, 0.98, 0.95)); inf.figura(fig)
    preds = [v for v in dict.fromkeys(d.Variable) if v not in res["objetivos"]]
    if preds:
        piv = lambda s, c: d[d.Subconjunto == s].set_index("Variable")[c]
        resumen = pd.DataFrame({"Variable": [etiqueta(v) for v in preds],
                                "Media ent.": piv("Entrenamiento", "Media").reindex(preds).values, "DE ent.": piv("Entrenamiento", "DE").reindex(preds).values,
                                "Media val.": piv("Validación", "Media").reindex(preds).values, "DE val.": piv("Validación", "DE").reindex(preds).values,
                                "Media prueba": piv("Prueba", "Media").reindex(preds).values, "DE prueba": piv("Prueba", "DE").reindex(preds).values,
                                "NA total": d[d.Variable.isin(preds)].groupby("Variable").n_NA.sum().reindex(preds).values})
        trozos = [resumen.iloc[i:i + 18] for i in range(0, len(resumen), 18)]
        for k, trozo in enumerate(trozos, 1):
            inf.salto(); inf.tabla(trozo, tam=8.5, decimales=3, titulo=f"Descriptivos de las variables predictoras ({k} de {len(trozos)})")

    # ---------------- 3. Resultados ----------------
    inf.salto(); inf.texto("3. Resultados", 16, True, VERDE, alto=0.42)
    t4, c4 = tabla4_df(res)
    t4.columns = ["Variable objetivo", "Modelo", "R²", "MAE", "RMSE", "MAPE (%)", "Red. MAE (%)", "Red. RMSE (%)", "Cumple"]
    inf.tabla(t4, c4, tam=8.5, titulo="Tabla 4. Desempeño predictivo de los modelos en el conjunto de prueba",
              nota="Nota. Fila verde = mejor modelo. Reducciones respecto al modelo base. Cumple: R² ≥ 0,75; MAPE ≤ 10 %; MAE y RMSE menores que los del modelo base.")
    inf.figura(fig_metricas(res, figsize=(11, 8.5)))
    for obj in res["objetivos"]:
        r = res["resultados"][obj]; num = NUMERACION.get(obj, {})
        inf.salto()
        inf.tabla(tabla_subconjunto_df(r), tam=9, decimales=4, titulo=f"Tabla {num.get('tabla', '')}. {etiqueta(obj)} por subconjunto",
                  nota=f"Modelo seleccionado: {r['mejor_modelo']} · Diebold-Mariano (HLN) RF vs. XGBoost: "
                       f"e² → DM = {fmt(r['dm']['estadistico'], 3)}, p = {fmt(r['dm']['p_valor'], 4)}; "
                       f"|e| → DM = {fmt(r['dm_abs']['estadistico'], 3)}, p = {fmt(r['dm_abs']['p_valor'], 4)}")
        alto_graf = inf.y - inf.MARGEN
        ax = inf.fig.add_axes([0.30, inf.MARGEN / 8.5, 0.44, (alto_graf - 0.75) / 8.5])
        dibujar_pred_real(ax, r, num_fig=num.get("pred"))
        inf.figura(fig_importancia(r, num.get("imp")))
    # Tablas 8 y 9: las mismas de la pestaña Resultados, con celdas combinadas
    s = res["sintesis"]; hg = s["general"]
    inf.salto()
    inf.tabla_hip(tabla8_mostrar(s["tabla8"]), COLS_T8, METRICAS_T8, FUSION_T8,
                  anchos=[1.55, 1.15, 0.85, 0.45, 0.7, 0.7, 0.7, 0.8, 0.95, 2.15], titulo=TITULO_TABLA8)
    inf.texto(f"Hipótesis general (Ho-Hi): {hg['decision']}.", 10, True, "#065F46" if hg["acepta"] else "#991B1B", alto=0.3)
    inf.texto(NOTA_TABLA8, 7.5, color="#4B5563", alto=0.32)
    inf.tabla_hip(tabla9_mostrar(s["tabla9"]), COLS_T9, METRICAS_T9, FUSION_T9,
                  anchos=[1.6, 1.15, 0.85, 0.45, 0.72, 0.72, 0.72, 0.8, 1.35, 1.95], titulo=TITULO_TABLA9)
    h4 = s["h4"]
    inf.texto(f"Hipótesis específica 4 (Ho4-Hi4): {h4['decision']}.", 10, True, "#065F46" if h4["acepta"] else "#991B1B", alto=0.3)
    inf.texto(NOTA_TABLA9, 7.5, color="#4B5563", alto=0.32)
    inf.cerrar()
    return ruta

## Paso 1 · Cargar el archivo de datos
Elige la fuente en el formulario de la derecha y ejecuta la celda:

* **Subir desde mi computadora**: aparece un botón *Elegir archivos* para seleccionar el Excel (`.xlsx`) o CSV.
* **Google Drive**: se pide autorización para montar tu Drive; escribe la ruta completa del archivo (por ejemplo `/content/drive/MyDrive/Tesis/Datos_Flotacion_de_Relaves.xlsx`).

In [ ]:
#@title Paso 1 · Cargar el archivo de datos { display-mode: "form" }
FUENTE = "Subir desde mi computadora"  #@param ["Subir desde mi computadora", "Google Drive"]
RUTA_DRIVE = "/content/drive/MyDrive/Datos_Flotacion_de_Relaves.xlsx"  #@param {type:"string"}

CARPETA_SALIDA = "/content" if EN_COLAB else os.getcwd()
if os.environ.get("RUTA_DATOS_PRUEBA"):                       # solo para pruebas fuera de Colab
    RUTA = os.environ["RUTA_DATOS_PRUEBA"]; NOMBRE_ARCHIVO = os.path.basename(RUTA)
    datos = leer_archivo_robusto(RUTA, NOMBRE_ARCHIVO)
elif FUENTE == "Subir desde mi computadora":
    if not EN_COLAB:
        raise RuntimeError("Esta opción requiere Google Colab. En Jupyter local usa 'Google Drive' con la ruta del archivo.")
    from google.colab import files
    subidos = files.upload()
    if not subidos: raise RuntimeError("No se seleccionó ningún archivo.")
    NOMBRE_ARCHIVO, contenido = next(iter(subidos.items()))
    datos = leer_archivo_robusto(io.BytesIO(contenido), NOMBRE_ARCHIVO)
elif FUENTE == "Google Drive":
    if EN_COLAB:
        from google.colab import drive
        drive.mount("/content/drive")
    if not os.path.exists(RUTA_DRIVE):
        candidatos = glob.glob("/content/drive/MyDrive/**/*.xlsx", recursive=True)[:15] if EN_COLAB else []
        raise FileNotFoundError(f"No existe: {RUTA_DRIVE}\nArchivos .xlsx encontrados en tu Drive:\n  " + "\n  ".join(candidatos))
    NOMBRE_ARCHIVO = os.path.basename(RUTA_DRIVE)
    datos = leer_archivo_robusto(RUTA_DRIVE, NOMBRE_ARCHIVO)
    CARPETA_SALIDA = os.path.dirname(RUTA_DRIVE)                 # el PDF se guardará junto al Excel
else:
    raise ValueError(f"Fuente no reconocida: {FUENTE}")

variables_detectadas = resumen_carga(datos, NOMBRE_ARCHIVO)
display(variables_detectadas)

## Paso 2 · Configurar y ejecutar el protocolo
Los valores por defecto reproducen el protocolo de la tesis. En `PREDICTORES_EXCLUIDOS` puedes escribir, separados por comas, los nombres de columna (columna *Columna* de la tabla anterior) que no quieras usar. Las dosificaciones de los cinco reactivos siempre se incluyen.

In [ ]:
#@title Paso 2 · Configurar y ejecutar el protocolo { display-mode: "form" }
OBJETIVOS = "recuperacion_au, ley_concentrado_au, consumo_reactivos"  #@param {type:"string"}
PREDICTORES_EXCLUIDOS = ""  #@param {type:"string"}
PROP_TRAIN = 0.7  #@param {type:"slider", min:0.5, max:0.9, step:0.05}
PROP_VALID = 0.15  #@param {type:"slider", min:0.05, max:0.3, step:0.05}
PARTICION_TEMPORAL = True  #@param {type:"boolean"}
USAR_COLUMNA_PARTICION = True  #@param {type:"boolean"}
MANEJO_NA = "media"  #@param ["media", "cero"]
USAR_HP_TESIS = True  #@param {type:"boolean"}

lista = lambda s: [x.strip() for x in s.split(",") if x.strip()]
numericas = columnas_numericas(datos)
objetivos = [o for o in lista(OBJETIVOS) if o in numericas]
faltan = [o for o in lista(OBJETIVOS) if o not in numericas]
if faltan: print("Aviso: no se encontraron estas variables objetivo:", faltan)
excluidos = set(lista(PREDICTORES_EXCLUIDOS))
pct_falt = datos[numericas].isna().mean()
predictores = [c for c in numericas if c not in objetivos and c not in excluidos and pct_falt[c] <= 0.5]

print(f"Objetivos: {[etiqueta(o) for o in objetivos]}")
print(f"Predictores seleccionados: {len(predictores)} (más dosificaciones y consumo total según el objetivo)")
inicio = datetime.now()
res = ejecutar_protocolo(datos, objetivos, predictores, PROP_TRAIN, PROP_VALID, PARTICION_TEMPORAL,
                         USAR_COLUMNA_PARTICION, MANEJO_NA, USAR_HP_TESIS)
print(f"Protocolo completado en {(datetime.now() - inicio).seconds} s · {res['particion']['origen']} "
      f"({res['particion']['n_train']} / {res['particion']['n_valid']} / {res['particion']['n_test']}).")
mostrar(html_tablas_hipotesis(res))

## Paso 3 · Ver resultados
Las pestañas reproducen las vistas de la app Shiny; haz clic en *Portada*, *Datos*, *Resultados*, *Discusión* o *Conclusiones*. Discusión y Conclusiones son texto fijo del informe (no se recalculan). En *Datos* puedes elegir la variable que quieres describir.

In [ ]:
#@title Paso 3 · Panel de resultados (Portada, Datos, Resultados, Discusión y Conclusiones)
mostrar_panel(res)

## Paso 4 · Descargar el informe PDF
Genera un PDF horizontal con las pestañas Portada, Datos y Resultados. Si cargaste el Excel desde Google Drive, el PDF también queda guardado en la misma carpeta.

In [ ]:
#@title Paso 4 · Generar y descargar el informe PDF
ruta_pdf = os.path.join(CARPETA_SALIDA, f"informe_comparacion_{datetime.now():%Y%m%d}.pdf")
generar_informe_pdf(res, ruta_pdf)
print("Informe generado:", ruta_pdf)
if EN_COLAB:
    from google.colab import files
    files.download(ruta_pdf)